## Troubleshooting — networking failures

### "Service unreachable from a pod" — the three-question flowchart

1. **Does DNS resolve?** From a test pod: `nslookup <svc>`. No → CoreDNS, or the pod's `dnsPolicy`/`resolv.conf`.
2. **Endpoints backed?** `kubectl get endpoints <svc>` — `<none>` = wrong selector or pods not `Ready`. `kubectl describe svc` shows the selector.
3. **Port reachable?** `nc -vz <svc> <port>`. No → NetworkPolicy or CNI.

### Other common cases

- **Right Service, wrong pod** — two Services with overlapping selectors, or `sessionAffinity` wrong for the protocol.
- **External traffic not arriving** — LoadBalancer `EXTERNAL-IP` still `<pending>` (LB controller hasn't provisioned); Ingress `Address` empty (controller isn't watching this `IngressClass`).
- **NetworkPolicy not blocking** — wrong CNI (**kindnet doesn't enforce**), no policy selects the pods (falls back to allow-all), or the `from`/`to` is too permissive (re-read OR-list / AND-fields, notebook 09).
- **`connection refused` from outside** — walk the layers: node IP reachable? NodePort open? selector matches pods? pods listening on the right port (`kubectl exec <pod> -- ss -ln`)? `containerPort` == `targetPort`?

### The netshoot move

```bash
kubectl run --rm tmp --image=nicolaka/netshoot -it -- bash
```

`netshoot` ships every tool — `dig`, `tcpdump`, `mtr`, `curl`, `nc`, `ss`, `ip` — so you can prod the cluster as a misbehaving pod would. On our map this frames the **Networking** box — where the three communication paths (notebook 09) meet real-world breakage, one layer at a time.